# 03a｜承接已完成预训练的金融领域训练

130U-FinLLM｜130U 金融大语言模型

本页先读取并校验 Drive 中至少完成扩充阶段 1220 步的检查点、数据和原 8K 词表，
将母检查点冻结后才允许训练。默认复用 `expansion_v1/domain_mix`，
其 1,215,545 tokens 包含约 80% 金融／20% 通用内容。

这是新的 `domain_v1` 阶段：首次只承接母模型权重，建立新的优化器和学习率调度，
阶段更新从 0 开始。已有本阶段检查点时严格恢复完整状态，不重新初始化。
选择「运行时 → 更改运行时类型 → T4 GPU」，从第一格顺序运行。

默认先验证第 1 步，再训练到本阶段累计 74 步，名义约 1,212,416 个 token 位置。
这是现有语料的一次有界小实验；不是 25M 领域预算已经完成，也不代表资产立场准确率。
不需要重做 01b、02c 或 02d，不要同时启动另一个训练进程。

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"缺少 {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} 不满足 {requirement}")
    if problems:
        raise RuntimeError("项目依赖检查失败：" + "; ".join(problems))
    print("项目已安装依赖检查通过。")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"缺少项目 Python 源码：{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"当前 Notebook 导入了错误的项目位置：{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("当前 Notebook 的检查点恢复接口不可用。")
    print("当前 Notebook 内核导入检查通过：", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("新 Python 进程导入检查通过：", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("PyTorch 环境意外变化，请保留输出并停止训练。")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook 环境准备完成；PyTorch 保持：", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
if not MOUNT_DRIVE:
    raise RuntimeError("本阶段需要已有种子输入及版本化备份，请完成 Drive 挂载。")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive 项目目录：", DRIVE_ROOT)

In [ ]:
DOMAIN_INPUTS_VERIFIED = False
DOMAIN_RETENTION_BEFORE_SUCCEEDED = False
DOMAIN_PROBE_SUCCEEDED = False
DOMAIN_TRAIN_SUCCEEDED = False
DOMAIN_RETENTION_AFTER_SUCCEEDED = False
DOMAIN_BACKED_UP = False
SEED_VERSION = "seed_v1"
EXPANSION_VERSION = "expansion_v1"
DOMAIN_VERSION = "domain_v1"
MIN_PARENT_UPDATES = 1220
PROBE_UPDATES = 1
TARGET_UPDATES = 74
if not torch.cuda.is_available():
    raise RuntimeError("请选择 T4 GPU 运行时；正式领域训练需要 GPU。")
from finllm.domain_runtime import (
    prepare_domain_training, run_domain_segment,
    retention_domain_check, backup_domain_training)
from finllm.checkpoint import load_checkpoint, sha256_file
domain = prepare_domain_training(
    PROJECT, DRIVE_ROOT, DOMAIN_VERSION, EXPANSION_VERSION, SEED_VERSION,
    min_parent_updates=MIN_PARENT_UPDATES, probe_updates=PROBE_UPDATES,
    target_updates=TARGET_UPDATES)
# 直接读已校验并冻结的母检查点，不把旧截图当成当前 Drive 读取结果。
parent_checkpoint = Path(domain["config"]["init_checkpoint"])
parent_payload = load_checkpoint(parent_checkpoint, allow_previous=False)
parent_audit = {
    "frozen_checkpoint": str(parent_checkpoint),
    "checkpoint_sha256": sha256_file(parent_checkpoint),
    "stage": parent_payload["config"]["stage"],
    "updates": parent_payload["progress"]["updates"],
    "tokens": parent_payload["progress"]["tokens"],
    "tokenizer_sha256": parent_payload["tokenizer_hash"],
    "data_sha256": parent_payload["provenance"]["data_sha256"],
    "validation_sha256": parent_payload["provenance"]["validation_sha256"],
}
if parent_audit["stage"] != "pretrain" or parent_audit["updates"] < MIN_PARENT_UPDATES:
    raise ValueError("母检查点不是要求的已完成预训练成果；保留已有文件并停止。")
print("当前已校验的 Drive 母检查点：", json.dumps(parent_audit, indent=2, ensure_ascii=False))
print("当前领域阶段累计步数：", domain["restored_updates"])
print("本阶段配置：", json.dumps(domain["config"], indent=2, ensure_ascii=False))
print("GPU:", torch.cuda.get_device_name(0))
DOMAIN_INPUTS_VERIFIED = True

上一格只检查输入、冻结母成果并准备配置，尚未启动优化器更新。
保留其实际母检查点 SHA256、步数和 tokens，作为本次云端读取证据。
下方诊断使用冻结的验证文本；诊断和训练 loss 衡量下一 token 预测，
不应解释为看多／看空分数。验证失败时后续训练和完整备份会停止。

In [ ]:
DOMAIN_RETENTION_BEFORE_SUCCEEDED = False
DOMAIN_PROBE_SUCCEEDED = False
DOMAIN_TRAIN_SUCCEEDED = False
DOMAIN_RETENTION_AFTER_SUCCEEDED = False
DOMAIN_BACKED_UP = False
if not globals().get("DOMAIN_INPUTS_VERIFIED", False):
    raise RuntimeError("请先完成当前 Drive 输入校验和母检查点冻结。")
domain_retention_before = retention_domain_check(domain, "before")
print(json.dumps(domain_retention_before, indent=2, ensure_ascii=False))
DOMAIN_RETENTION_BEFORE_SUCCEEDED = True

In [ ]:
DOMAIN_PROBE_SUCCEEDED = False
DOMAIN_TRAIN_SUCCEEDED = False
DOMAIN_RETENTION_AFTER_SUCCEEDED = False
DOMAIN_BACKED_UP = False
if not globals().get("DOMAIN_INPUTS_VERIFIED", False) or not globals().get("DOMAIN_RETENTION_BEFORE_SUCCEEDED", False):
    raise RuntimeError("请先完成母检查点校验和训练前诊断。")
domain_probe = run_domain_segment(domain, "probe", finllm)
print(json.dumps(domain_probe, indent=2, ensure_ascii=False))
DOMAIN_PROBE_SUCCEEDED = True

单步成功后继续到本阶段累计 74 步。母预训练与领域阶段的 tokens 分别记录，
沿模型路径的累计量另列。原 seed、扩充训练、1220 步母检查点及其报告保持原样。
首次领域权重承接与后续严格恢复分别处理；数据、词表、配置或本阶段运行时不兼容会停止。
连接断开后先确认旧执行状态，再从本页第一格恢复，保留已有检查点和报告。

In [ ]:
DOMAIN_TRAIN_SUCCEEDED = False
DOMAIN_RETENTION_AFTER_SUCCEEDED = False
DOMAIN_BACKED_UP = False
if not globals().get("DOMAIN_PROBE_SUCCEEDED", False):
    raise RuntimeError("请先成功完成领域阶段单步验证。")
domain_result = run_domain_segment(domain, "continue", finllm)
print(json.dumps(domain_result, indent=2, ensure_ascii=False))
DOMAIN_TRAIN_SUCCEEDED = True

In [ ]:
DOMAIN_RETENTION_AFTER_SUCCEEDED = False
DOMAIN_BACKED_UP = False
if not globals().get("DOMAIN_TRAIN_SUCCEEDED", False):
    raise RuntimeError("本阶段训练尚未成功结束；保留已有备份。")
domain_retention_after = retention_domain_check(domain, "after")
print(json.dumps(domain_retention_after, indent=2, ensure_ascii=False))
DOMAIN_RETENTION_AFTER_SUCCEEDED = True

In [ ]:
DOMAIN_BACKED_UP = False
if not all(globals().get(flag, False) for flag in (
    "DOMAIN_INPUTS_VERIFIED", "DOMAIN_RETENTION_BEFORE_SUCCEEDED",
    "DOMAIN_PROBE_SUCCEEDED", "DOMAIN_TRAIN_SUCCEEDED", "DOMAIN_RETENTION_AFTER_SUCCEEDED")):
    raise RuntimeError("输入／诊断／训练尚未全部成功；不得显示完整阶段备份。")
domain_backup = backup_domain_training(domain)
print(json.dumps(domain_backup, indent=2, ensure_ascii=False))
shutil.copy2(PROJECT / "runtime_revision.txt", domain["paths"]["drive"] / "runtime_revision.txt")
DOMAIN_BACKED_UP = True
print("金融领域训练与校验备份完成：", domain["paths"]["drive"] / "run")
print("领域阶段报告与诊断：", domain["paths"]["drive"])
print("完整母检查点已冻结：", parent_checkpoint)

完成标准：实际母检查点身份已读取；领域单步与累计 74 步报告可核验；
训练前后诊断保留；最后备份为 `verified: true`。
已完成阶段重复运行会复用原检查点与真实测量，不重新计时或覆盖原报告。
本页完成有限金融语料实验，原 100M／25M 计算预算仍单独记录。
正式 SFT、DPO、校准和资产立场评测继续需要符合本项目定义的目标级标签与审核记录。